<a href="https://colab.research.google.com/github/hamedhossam22/repo-Multimodal-Review-Consistency-Detection/blob/main/moramadan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 Import DistilBERT Components

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification
)

In [ ]:
# Load DistilBERT Tokenizer

distilbert_tokenizer = DistilBertTokenizer.from_pretrained(
    'distilbert-base-uncased'
)

print("DistilBERT tokenizer loaded successfully!")

In [ ]:
# Encode Reviews for DistilBERT

distilbert_inputs = distilbert_tokenizer(
    df['cleaned_text'].tolist(),
    padding=True,
    truncation=True,
    max_length=100,
    return_tensors='pt'
)

print("DistilBERT encoding completed successfully!")

In [ ]:
# Inspect DistilBERT Input Shapes

print(
    "Input IDs Shape:",
    distilbert_inputs['input_ids'].shape
)

print(
    "Attention Mask Shape:",
    distilbert_inputs['attention_mask'].shape
)

In [ ]:
# Load DistilBERT Classifier

distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

print("DistilBERT model loaded successfully!")

In [ ]:
# Move DistilBERT to Device

distilbert_model.to(device)

print("DistilBERT moved to device successfully!")

In [ ]:
# Configure DistilBERT Optimizer

distilbert_optimizer = AdamW(
    distilbert_model.parameters(),
    lr=2e-5
)

print("DistilBERT optimizer configured successfully!")

In [ ]:
# DistilBERT Training Loop

from tqdm import tqdm

distilbert_model.train()

for epoch in range(2):

    total_loss = 0

    print(f"\nEpoch {epoch+1}")

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        batch_input_ids = batch[0].to(device)
        batch_attention_masks = batch[1].to(device)
        batch_labels = batch[2].to(device)

        distilbert_optimizer.zero_grad()

        outputs = distilbert_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_masks,
            labels=batch_labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        distilbert_optimizer.step()

        progress_bar.set_postfix({'Loss': loss.item()})

    avg_loss = total_loss / len(train_loader)

    print("Average Training Loss:", avg_loss)

In [ ]:
# Evaluate DistilBERT

distilbert_model.eval()

distil_predictions = []
distil_true_labels = []

with torch.no_grad():

    for batch in test_loader:

        batch_input_ids = batch[0].to(device)
        batch_attention_masks = batch[1].to(device)
        batch_labels = batch[2].to(device)

        outputs = distilbert_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_masks
        )

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        distil_predictions.extend(preds.cpu().numpy())
        distil_true_labels.extend(batch_labels.cpu().numpy())

In [ ]:
# DistilBERT Accuracy

distil_accuracy = accuracy_score(
    distil_true_labels,
    distil_predictions
)

print("DistilBERT Test Accuracy:", distil_accuracy)

In [ ]:
# DistilBERT Classification Report

print(
    classification_report(
        distil_true_labels,
        distil_predictions
    )
)

In [ ]:
# DistilBERT Confusion Matrix

distil_cm = confusion_matrix(
    distil_true_labels,
    distil_predictions
)

print(distil_cm)

In [ ]:
# Final Model Comparison

comparison_df = pd.DataFrame({

    'Model': [
        'LSTM',
        'BERT',
        'DistilBERT'
    ],

    'Architecture Type': [
        'Traditional Sequence Model',
        'Transformer Model',
        'Lightweight Transformer'
    ],

    'Test Accuracy': [
        0.744,
        0.941,
        0.919
    ],

    'Training Speed': [
        'Medium',
        'Slow',
        'Fast'
    ],

    'Contextual Understanding': [
        'Limited',
        'Excellent',
        'Very Strong'
    ]
})

comparison_df